### Import Dependencies

In [115]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.model_selection import KFold
from tqdm.notebook import tqdm
import mlflow
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from lightgbm import early_stopping, log_evaluation
from catboost import CatBoostRegressor
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    r2_score, 
    mean_absolute_error, 
    root_mean_squared_error
)

pd.set_option('display.max_columns', None)

### Data Preparation

In [116]:
# NASA S-metric: Measures the practical cost of prediction errors
# with a higher penalty for dangerous overestimation of remaining life.
def s_score(
    rul_true: float,
    rul_pred: float,
) -> float:
    """
    Compute the NASA C-MAPSS S-score.

    Parameters
    ----------
    rul_true : np.ndarray
        Ground truth Remaining Useful Life (RUL).
    rul_pred : np.ndarray
        Predicted Remaining Useful Life (RUL).

    Returns
    -------
    float
        Total NASA S-score. Lower is better.
    """

    diff = rul_pred - rul_true

    score = np.where(
        diff < 0,
        np.exp(-diff / 13.0) - 1.0,
        np.exp(diff / 10.0) - 1.0,
    )

    return float(np.sum(score))

In [117]:
# # reading data from source
# train = pd.read_csv(r'..\data\train_FD001.txt', sep=r'\s+', header=None)
# test = pd.read_csv(r'..\data\test_FD001.txt', sep=r'\s+', header=None)
# test_labels = pd.read_csv(r'..\data\RUL_FD001.txt', header=None)

# # feature names as written in readme file
# columns = (
#     ["unit_number", "time_cycles"]
#     + [f"operational_setting_{i}" for i in range(1, 4)]
#     + [f"sensor_measurement_{i}" for i in range(1, 22)]
# )

# # renaming features
# train.columns = columns
# test.columns = columns
# test_labels.columns = ['rul']

# # maximum cycle for each engine unit
# max_cycles = train.groupby('unit_number')['time_cycles'].max().reset_index()
# max_cycles.columns = ['unit_number', 'max_cycle']

# # merge this maximum back into the main dataframe
# train = train.merge(max_cycles, on='unit_number', how='left')

# # calculate the standard linear RUL
# train['linear_rul'] = train['max_cycle'] - train['time_cycles']

# # drop the helper column
# train.drop(columns=['max_cycle'], inplace=True)

# # industry standard cap for FD001
# R_max = 120

# # apply the ceiling cap: if linear_rul > R_max, force it to be R_max
# train['piecewise_rul'] = train['linear_rul'].clip(upper=R_max)

In [118]:
# train-test split
X = pd.read_csv(r'..\notebooks\nb_artifacts\X_fe.csv')
X_test = pd.read_csv(r'..\notebooks\nb_artifacts\X_test_fe.csv')

y_linear = pd.read_csv(r'..\notebooks\nb_artifacts\y_linear.csv')
y_linear = y_linear.iloc[:, 0]

y_piecewise = pd.read_csv(r'..\notebooks\nb_artifacts\y_piecewise.csv')
y_piecewise = y_piecewise.iloc[:, 0]

y_test = pd.read_csv(r'..\notebooks\nb_artifacts\y_test.csv')
y_test = y_test.iloc[:, 0]

print(f'Shape of X: {X.shape}')
print(f'Shape of y_linear: {y_linear.shape}')
print(f'Shape of y_piecewise: {y_piecewise.shape}')
print(f'Shape of X_test: {X_test.shape}')
print(f'Shape of y_test: {y_test.shape}')

Shape of X: (20631, 49)
Shape of y_linear: (20631,)
Shape of y_piecewise: (20631,)
Shape of X_test: (100, 49)
Shape of y_test: (100,)


### Configuration

In [119]:
class Config:

    SEED = 42
    N_SPLITS = 5

    TARGET_TYPE = 'piecewise_rul'

    HISTGBM_PARAMS = {
        'categorical_features': 'from_dtype',
        'early_stopping': 'auto',
        'interaction_cst': None,
        'l2_regularization': 0.0,
        'learning_rate': 0.1,
        'loss': 'squared_error',
        'max_bins': 255,
        'max_depth': 5,
        'max_features': 0.8,
        'max_iter': 400,
        'max_leaf_nodes': 31,
        'min_samples_leaf': 20,
        'monotonic_cst': None,
        'n_iter_no_change': 10,
        'quantile': None,
        'random_state': SEED,
        'scoring': 'loss',
        'tol': 1e-07,
        'validation_fraction': None,
        'verbose': 0,
        'warm_start': False
    }

    ETREES_PARAMS = {
        'bootstrap': True,
        'ccp_alpha': 0.0,
        'criterion': 'squared_error',
        'max_depth': None,
        'max_features': 1.0,
        'max_leaf_nodes': None,
        'max_samples': None,
        'min_impurity_decrease': 0.0,
        'min_samples_leaf': 1,
        'min_samples_split': 2,
        'min_weight_fraction_leaf': 0.0,
        'monotonic_cst': None,
        'n_estimators': 400,
        'n_jobs': -1,
        'oob_score': False,
        'random_state': SEED,
        'verbose': 0,
        'warm_start': False
    }

    XGB_PARAMS = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'tree_method': 'hist',
        'device': 'cuda',
        'learning_rate': 0.05,
        'n_estimators': 1200,
        'max_depth': 5,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'random_state': SEED,
        'n_jobs': -1,
        'verbosity': 0,
        'early_stopping_rounds': 100,
        'min_child_weight': 3, 
        'reg_alpha': 0.1, 
        'reg_lambda': 1.0, 
        'gamma': 0.1
    }

    LGBM_PARAMS = {
        "boosting_type": "gbdt",
        "data_sample_strategy": "goss",
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.03,
        "n_estimators": 500,
        "num_leaves": 31,
        "max_depth": -1,
        "colsample_bytree": 1.0,
        "reg_alpha": 1.0,
        "reg_lambda": 0.5,
        "random_state": SEED,
        "importance_type": "gain",
        "n_jobs": -1,
    }

TARGETS = {
    "linear_rul": y_linear,
    "piecewise_rul": y_piecewise,
}

y = TARGETS[Config.TARGET_TYPE]

VERSION = "V8"

### Common utils

In [120]:
def plot_permutation_importance(
    model,
    X: pd.DataFrame,
    y,
    scoring: str = "neg_mean_squared_error",
    n_repeats: int = 10,
    top_k: int | None = None,
    random_state: int = 42,
    figsize: tuple = (10, 8),
) -> None:
    """
    Plot permutation feature importance.

    Parameters
    ----------
    model : estimator
        Trained scikit-learn model.

    X : pd.DataFrame
        Feature matrix.

    y : array-like
        Target values.

    scoring : str, default="neg_mean_squared_error"
        Scoring metric used during permutation importance.

    n_repeats : int, default=20
        Number of feature shuffles.

    top_k : int or None, default=None
        Number of top features to display.
        If None, all features are plotted.

    random_state : int, default=42
        Random seed.

    figsize : tuple, default=(10, 8)
        Figure size.
    """

    perm = permutation_importance(
        estimator=model,
        X=X,
        y=y,
        scoring=scoring,
        n_repeats=n_repeats,
        random_state=random_state,
        n_jobs=-1,
    )

    importance_df = (
        pd.DataFrame(
            {
                "Feature": X.columns,
                "Importance": perm.importances_mean,
                "Std": perm.importances_std,
            }
        )
        .sort_values("Importance", ascending=False)
    )

    if top_k is not None:
        importance_df = importance_df.head(top_k)

    fig, ax = plt.subplots(figsize=figsize)

    ax.barh(
        importance_df["Feature"],
        importance_df["Importance"],
        xerr=importance_df["Std"],
        color="#5B8FF9",
        edgecolor="black",
    )

    ax.invert_yaxis()

    ax.set_title(
        "Permutation Feature Importance",
        fontsize=18,
        fontweight="bold",
        pad=15,
    )

    ax.set_xlabel(
        "Mean Importance",
        fontsize=12,
    )

    ax.set_ylabel("")

    ax.grid(
        axis="x",
        linestyle="--",
        alpha=0.3,
    )

    sns.despine()

    plt.tight_layout()

    return fig

In [121]:
def plot_actual_vs_predicted_scatter(
    y_true,
    y_pred,
    title: str = "Actual vs Predicted",
    figsize: tuple = (7, 7),
):
    """
    Plot Actual vs Predicted scatter plot.

    Parameters
    ----------
    y_true : array-like
        Ground truth values.

    y_pred : array-like
        Predicted values.

    title : str, default="Actual vs Predicted"
        Plot title.

    figsize : tuple, default=(7, 7)
        Figure size.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Matplotlib figure.
    """

    fig, ax = plt.subplots(figsize=figsize)

    sns.scatterplot(
        x=y_true,
        y=y_pred,
        s=30,
        alpha=0.6,
        edgecolor=None,
        ax=ax,
    )

    # Perfect prediction line
    lims = [
        min(np.min(y_true), np.min(y_pred)),
        max(np.max(y_true), np.max(y_pred)),
    ]

    ax.plot(
        lims,
        lims,
        linestyle="--",
        color="red",
        linewidth=2,
        label="Perfect Prediction",
    )

    ax.set_xlim(lims)
    ax.set_ylim(lims)

    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold",
        pad=15,
    )

    ax.set_xlabel(
        "Actual Remaining Useful Life",
        fontsize=12,
    )

    ax.set_ylabel(
        "Predicted Remaining Useful Life",
        fontsize=12,
    )

    ax.grid(
        alpha=0.3,
        linestyle="--",
    )

    ax.legend()

    sns.despine()

    plt.tight_layout()

    return fig

## Training and Experiment Tracking

### 1. HistGradientBoostingRegressor

In [122]:
# MLflow
EXPERIMENT_NAME = "histgbm"

RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{Config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# Out-of-fold and Test predictions
oof_predictions = np.zeros(len(X), dtype=np.float32)
test_predictions = np.zeros(len(X_test), dtype=np.float32)

# Cross Validation
kf = KFold(
    n_splits=Config.N_SPLITS,
    shuffle=True,
    random_state=Config.SEED,
)

with mlflow.start_run(run_name=RUN_NAME):

    # Log parameters
    mlflow.log_params(Config.HISTGBM_PARAMS)

    mlflow.log_params(
        {
            "model": "HistGradientBoostingRegressor",
            "target_type": Config.TARGET_TYPE,
            "n_splits": Config.N_SPLITS,
            "seed": Config.SEED,
            "n_features": X.shape[1],
            "n_samples": X.shape[0]
        }
    )

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(kf.split(X, y), start=1),
        total=Config.N_SPLITS,
        desc="Training",
    ):

        # Split
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        # Model
        model = HistGradientBoostingRegressor(
            **Config.HISTGBM_PARAMS
        )

        # Train
        model.fit(X_train, y_train)

        # Predictions
        valid_pred = model.predict(X_valid)
        test_pred = model.predict(X_test)

        # Save predictions
        oof_predictions[valid_idx] = valid_pred
        test_predictions += test_pred / Config.N_SPLITS

        # Metrics
        rmse = root_mean_squared_error(
            y_valid,
            valid_pred,
        )

        mae = mean_absolute_error(
            y_valid,
            valid_pred,
        )

        r2 = r2_score(
            y_valid,
            valid_pred,
        )

        s_metric = s_score(
            y_valid.to_numpy(),
            valid_pred,
        )

        print(
            f"Fold {fold:02d} | "
            f"RMSE: {rmse:.4f} | "
            f"MAE: {mae:.4f} | "
            f"R2 score: {r2:.4f} | "
            f"S-score: {s_metric:.4f}"
        )

    print("\n" + "=" * 70)
    print("OOF RESULTS")
    print("=" * 70)

    oof_rmse = root_mean_squared_error(
        y,
        oof_predictions,
    )

    oof_mae = mean_absolute_error(
        y,
        oof_predictions,
    )

    oof_r2 = r2_score(
        y,
        oof_predictions,
    )

    oof_s = s_score(
        y.to_numpy(),
        oof_predictions,
    )

    print(f"RMSE: {oof_rmse:.4f}")
    print(f"MAE: {oof_mae:.4f}")
    print(f"R2 score: {oof_r2:.4f}")
    print(f"S-score: {oof_s:.2f}")

    # Log OOF metrics
    mlflow.log_metrics(
        {
            "oof_rmse": oof_rmse,
            "oof_mae": oof_mae,
            "oof_r2_score": oof_r2,
            "oof_s_score": oof_s,
        }
    )

    print("\n" + "=" * 70)
    print("TEST RESULTS")
    print("=" * 70)

    test_rmse = root_mean_squared_error(
        y_test,
        test_predictions,
    )

    test_mae = mean_absolute_error(
        y_test,
        test_predictions,
    )

    test_r2 = r2_score(
        y_test,
        test_predictions,
    )

    test_s = s_score(
        y_test.to_numpy(),
        test_predictions,
    )

    print(f"RMSE: {test_rmse:.4f}")
    print(f"MAE: {test_mae:.4f}")
    print(f"R2 score: {test_r2:.4f}")
    print(f"S-score: {test_s:.2f}")

    # Log test metrics
    mlflow.log_metrics(
        {
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "test_s_score": test_s,
        }
    )

    print('Logging Plots')
    fig = plot_permutation_importance(
        model=model,
        X=X_valid,
        y=y_valid,
        top_k=15,
    )

    mlflow.log_figure(
        fig,
        "plots/permutation_importance.png",
    )

    plt.close(fig)

    fig = plot_actual_vs_predicted_scatter(
        y_true=y,
        y_pred=oof_predictions,
        title="OOF Actual vs Predicted"
    )

    mlflow.log_figure(
        fig,
        "plots/oof_actual_vs_predicted.png"
    )

    plt.close(fig)

    print('Logging Done')

Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 01 | RMSE: 15.2504 | MAE: 10.6153 | R2 score: 0.8500 | S-score: 35168.1864
Fold 02 | RMSE: 14.9563 | MAE: 10.4908 | R2 score: 0.8595 | S-score: 30377.9360
Fold 03 | RMSE: 14.9730 | MAE: 10.5138 | R2 score: 0.8571 | S-score: 27182.5906
Fold 04 | RMSE: 14.6510 | MAE: 10.2676 | R2 score: 0.8660 | S-score: 25171.9025
Fold 05 | RMSE: 15.1260 | MAE: 10.5688 | R2 score: 0.8594 | S-score: 28349.2776

OOF RESULTS
RMSE: 14.9927
MAE: 10.4913
R2 score: 0.8585
S-score: 146249.89

TEST RESULTS
RMSE: 18.3165
MAE: 13.3151
R2 score: 0.8057
S-score: 759.25
Logging Plots
Logging Done


### 2. ExtraTreesRegressor

In [123]:
# MLflow
EXPERIMENT_NAME = "etrees"

RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{Config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# Out-of-fold and Test predictions
oof_predictions = np.zeros(len(X), dtype=np.float32)
test_predictions = np.zeros(len(X_test), dtype=np.float32)

# Cross Validation
kf = KFold(
    n_splits=Config.N_SPLITS,
    shuffle=True,
    random_state=Config.SEED,
)

with mlflow.start_run(run_name=RUN_NAME):

    # Log parameters
    mlflow.log_params(Config.ETREES_PARAMS)

    mlflow.log_params(
        {
            "model": "ExtraTreesRegressor",
            "target_type": Config.TARGET_TYPE,
            "n_splits": Config.N_SPLITS,
            "seed": Config.SEED,
            "n_features": X.shape[1],
            "n_samples": X.shape[0]
        }
    )

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(kf.split(X, y), start=1),
        total=Config.N_SPLITS,
        desc="Training",
    ):

        # Split
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        # Model
        model = ExtraTreesRegressor(
            **Config.ETREES_PARAMS
        )

        # Train
        model.fit(X_train, y_train)

        # Predictions
        valid_pred = model.predict(X_valid)
        test_pred = model.predict(X_test)

        # Save predictions
        oof_predictions[valid_idx] = valid_pred
        test_predictions += test_pred / Config.N_SPLITS

        # Metrics
        rmse = root_mean_squared_error(
            y_valid,
            valid_pred,
        )

        mae = mean_absolute_error(
            y_valid,
            valid_pred,
        )

        r2 = r2_score(
            y_valid,
            valid_pred,
        )

        s_metric = s_score(
            y_valid.to_numpy(),
            valid_pred,
        )

        print(
            f"Fold {fold:02d} | "
            f"RMSE: {rmse:.4f} | "
            f"MAE: {mae:.4f} | "
            f"R2 score: {r2:.4f} | "
            f"S-score: {s_metric:.4f}"
        )

    print("\n" + "=" * 70)
    print("OOF RESULTS")
    print("=" * 70)

    oof_rmse = root_mean_squared_error(
        y,
        oof_predictions,
    )

    oof_mae = mean_absolute_error(
        y,
        oof_predictions,
    )

    oof_r2 = r2_score(
        y,
        oof_predictions,
    )

    oof_s = s_score(
        y.to_numpy(),
        oof_predictions,
    )

    print(f"RMSE: {oof_rmse:.4f}")
    print(f"MAE: {oof_mae:.4f}")
    print(f"R2 score: {oof_r2:.4f}")
    print(f"S-score: {oof_s:.2f}")

    # Log OOF metrics
    mlflow.log_metrics(
        {
            "oof_rmse": oof_rmse,
            "oof_mae": oof_mae,
            "oof_r2_score": oof_r2,
            "oof_s_score": oof_s,
        }
    )

    print("\n" + "=" * 70)
    print("TEST RESULTS")
    print("=" * 70)

    test_rmse = root_mean_squared_error(
        y_test,
        test_predictions,
    )

    test_mae = mean_absolute_error(
        y_test,
        test_predictions,
    )

    test_r2 = r2_score(
        y_test,
        test_predictions,
    )

    test_s = s_score(
        y_test.to_numpy(),
        test_predictions,
    )

    print(f"RMSE: {test_rmse:.4f}")
    print(f"MAE: {test_mae:.4f}")
    print(f"R2 score: {test_r2:.4f}")
    print(f"S-score: {test_s:.2f}")

    # Log test metrics
    mlflow.log_metrics(
        {
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "test_s_score": test_s,
        }
    )

    print('Logging Plots')
    fig = plot_permutation_importance(
        model=model,
        X=X_valid,
        y=y_valid,
        top_k=15,
    )

    mlflow.log_figure(
        fig,
        "plots/permutation_importance.png",
    )

    plt.close(fig)

    fig = plot_actual_vs_predicted_scatter(
        y_true=y,
        y_pred=oof_predictions,
        title="OOF Actual vs Predicted"
    )

    mlflow.log_figure(
        fig,
        "plots/oof_actual_vs_predicted.png"
    )

    plt.close(fig)

    print('Logging Done')

Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 01 | RMSE: 15.7869 | MAE: 11.1276 | R2 score: 0.8393 | S-score: 35782.0561
Fold 02 | RMSE: 15.4754 | MAE: 11.0549 | R2 score: 0.8496 | S-score: 31479.7667
Fold 03 | RMSE: 15.5480 | MAE: 11.0914 | R2 score: 0.8459 | S-score: 32193.9367
Fold 04 | RMSE: 15.0721 | MAE: 10.6910 | R2 score: 0.8581 | S-score: 26141.3290
Fold 05 | RMSE: 15.6704 | MAE: 11.0416 | R2 score: 0.8491 | S-score: 34395.5367

OOF RESULTS
RMSE: 15.5125
MAE: 11.0013
R2 score: 0.8485
S-score: 159992.63

TEST RESULTS
RMSE: 18.0659
MAE: 13.0668
R2 score: 0.8110
S-score: 864.24
Logging Plots
Logging Done


### 3. XGBRegressor

In [124]:
# MLflow
EXPERIMENT_NAME = "xgboost"

RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{Config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# Out-of-fold and Test predictions
oof_predictions = np.zeros(len(X), dtype=np.float32)
test_predictions = np.zeros(len(X_test), dtype=np.float32)

# Cross Validation
kf = KFold(
    n_splits=Config.N_SPLITS,
    shuffle=True,
    random_state=Config.SEED,
)

with mlflow.start_run(run_name=RUN_NAME):

    # Log parameters
    mlflow.log_params(Config.XGB_PARAMS)

    mlflow.log_params(
        {
            "model": "XGBRegressor",
            "target_type": Config.TARGET_TYPE,
            "n_splits": Config.N_SPLITS,
            "seed": Config.SEED,
            "n_features": X.shape[1],
            "n_samples": X.shape[0]
        }
    )

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(kf.split(X, y), start=1),
        total=Config.N_SPLITS,
        desc="Training",
    ):

        # Split
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        # Model
        model = XGBRegressor(
            **Config.XGB_PARAMS
        )

        # Train
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=0
        )

        # Predictions
        valid_pred = model.predict(X_valid)
        test_pred = model.predict(X_test)

        # Save predictions
        oof_predictions[valid_idx] = valid_pred
        test_predictions += test_pred / Config.N_SPLITS

        # Metrics
        rmse = root_mean_squared_error(
            y_valid,
            valid_pred,
        )

        mae = mean_absolute_error(
            y_valid,
            valid_pred,
        )

        r2 = r2_score(
            y_valid,
            valid_pred,
        )

        s_metric = s_score(
            y_valid.to_numpy(),
            valid_pred,
        )

        print(
            f"Fold {fold:02d} | "
            f"RMSE: {rmse:.4f} | "
            f"MAE: {mae:.4f} | "
            f"R2 score: {r2:.4f} | "
            f"S-score: {s_metric:.4f}"
        )

    print("\n" + "=" * 70)
    print("OOF RESULTS")
    print("=" * 70)

    oof_rmse = root_mean_squared_error(
        y,
        oof_predictions,
    )

    oof_mae = mean_absolute_error(
        y,
        oof_predictions,
    )

    oof_r2 = r2_score(
        y,
        oof_predictions,
    )

    oof_s = s_score(
        y.to_numpy(),
        oof_predictions,
    )

    print(f"RMSE: {oof_rmse:.4f}")
    print(f"MAE: {oof_mae:.4f}")
    print(f"R2 score: {oof_r2:.4f}")
    print(f"S-score: {oof_s:.2f}")

    # Log OOF metrics
    mlflow.log_metrics(
        {
            "oof_rmse": oof_rmse,
            "oof_mae": oof_mae,
            "oof_r2_score": oof_r2,
            "oof_s_score": oof_s,
        }
    )

    print("\n" + "=" * 70)
    print("TEST RESULTS")
    print("=" * 70)

    test_rmse = root_mean_squared_error(
        y_test,
        test_predictions,
    )

    test_mae = mean_absolute_error(
        y_test,
        test_predictions,
    )

    test_r2 = r2_score(
        y_test,
        test_predictions,
    )

    test_s = s_score(
        y_test.to_numpy(),
        test_predictions,
    )

    print(f"RMSE: {test_rmse:.4f}")
    print(f"MAE: {test_mae:.4f}")
    print(f"R2 score: {test_r2:.4f}")
    print(f"S-score: {test_s:.2f}")

    # Log test metrics
    mlflow.log_metrics(
        {
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "test_s_score": test_s,
        }
    )

    print('Logging Plots')
    fig = plot_permutation_importance(
        model=model,
        X=X_valid,
        y=y_valid,
        top_k=15,
    )

    mlflow.log_figure(
        fig,
        "plots/permutation_importance.png",
    )

    plt.close(fig)

    fig = plot_actual_vs_predicted_scatter(
        y_true=y,
        y_pred=oof_predictions,
        title="OOF Actual vs Predicted"
    )

    mlflow.log_figure(
        fig,
        "plots/oof_actual_vs_predicted.png"
    )

    plt.close(fig)

    print('Logging Done')

Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 01 | RMSE: 15.0536 | MAE: 10.4407 | R2 score: 0.8539 | S-score: 34050.9387
Fold 02 | RMSE: 14.7441 | MAE: 10.3096 | R2 score: 0.8635 | S-score: 27975.3346
Fold 03 | RMSE: 14.6944 | MAE: 10.3571 | R2 score: 0.8624 | S-score: 25249.6435
Fold 04 | RMSE: 14.4274 | MAE: 10.0566 | R2 score: 0.8700 | S-score: 24024.7507
Fold 05 | RMSE: 14.9092 | MAE: 10.3607 | R2 score: 0.8634 | S-score: 28822.7308

OOF RESULTS
RMSE: 14.7673
MAE: 10.3049
R2 score: 0.8627
S-score: 140123.40

TEST RESULTS
RMSE: 18.1463
MAE: 13.0391
R2 score: 0.8093
S-score: 774.51
Logging Plots
Logging Done


### 4. LGBMRegressor

In [125]:
# MLflow
EXPERIMENT_NAME = "lgbm"

RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{Config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# Out-of-fold and Test predictions
oof_predictions = np.zeros(len(X), dtype=np.float32)
test_predictions = np.zeros(len(X_test), dtype=np.float32)

# Cross Validation
kf = KFold(
    n_splits=Config.N_SPLITS,
    shuffle=True,
    random_state=Config.SEED,
)

with mlflow.start_run(run_name=RUN_NAME):

    # Log parameters
    mlflow.log_params(Config.LGBM_PARAMS)

    mlflow.log_params(
        {
            "model": "LGBMRegressor",
            "target_type": Config.TARGET_TYPE,
            "n_splits": Config.N_SPLITS,
            "seed": Config.SEED,
            "n_features": X.shape[1],
            "n_samples": X.shape[0]
        }
    )

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(kf.split(X, y), start=1),
        total=Config.N_SPLITS,
        desc="Training",
    ):

        # Split
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        # Model
        model = LGBMRegressor(
            **Config.LGBM_PARAMS
        )

        # Train
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            callbacks=[early_stopping(100, verbose=False)]
        )

        # Predictions
        valid_pred = model.predict(X_valid)
        test_pred = model.predict(X_test)

        # Save predictions
        oof_predictions[valid_idx] = valid_pred
        test_predictions += test_pred / Config.N_SPLITS

        # Metrics
        rmse = root_mean_squared_error(
            y_valid,
            valid_pred,
        )

        mae = mean_absolute_error(
            y_valid,
            valid_pred,
        )

        r2 = r2_score(
            y_valid,
            valid_pred,
        )

        s_metric = s_score(
            y_valid.to_numpy(),
            valid_pred,
        )

        print(
            f"Fold {fold:02d} | "
            f"RMSE: {rmse:.4f} | "
            f"MAE: {mae:.4f} | "
            f"R2 score: {r2:.4f} | "
            f"S-score: {s_metric:.4f}"
        )

    print("\n" + "=" * 70)
    print("OOF RESULTS")
    print("=" * 70)

    oof_rmse = root_mean_squared_error(
        y,
        oof_predictions,
    )

    oof_mae = mean_absolute_error(
        y,
        oof_predictions,
    )

    oof_r2 = r2_score(
        y,
        oof_predictions,
    )

    oof_s = s_score(
        y.to_numpy(),
        oof_predictions,
    )

    print(f"RMSE: {oof_rmse:.4f}")
    print(f"MAE: {oof_mae:.4f}")
    print(f"R2 score: {oof_r2:.4f}")
    print(f"S-score: {oof_s:.2f}")

    # Log OOF metrics
    mlflow.log_metrics(
        {
            "oof_rmse": oof_rmse,
            "oof_mae": oof_mae,
            "oof_r2_score": oof_r2,
            "oof_s_score": oof_s,
        }
    )

    print("\n" + "=" * 70)
    print("TEST RESULTS")
    print("=" * 70)

    test_rmse = root_mean_squared_error(
        y_test,
        test_predictions,
    )

    test_mae = mean_absolute_error(
        y_test,
        test_predictions,
    )

    test_r2 = r2_score(
        y_test,
        test_predictions,
    )

    test_s = s_score(
        y_test.to_numpy(),
        test_predictions,
    )

    print(f"RMSE: {test_rmse:.4f}")
    print(f"MAE: {test_mae:.4f}")
    print(f"R2 score: {test_r2:.4f}")
    print(f"S-score: {test_s:.2f}")

    # Log test metrics
    mlflow.log_metrics(
        {
            "test_rmse": test_rmse,
            "test_mae": test_mae,
            "test_r2": test_r2,
            "test_s_score": test_s,
        }
    )

    print('Logging Plots')
    fig = plot_permutation_importance(
        model=model,
        X=X_valid,
        y=y_valid,
        top_k=15,
    )

    mlflow.log_figure(
        fig,
        "plots/permutation_importance.png",
    )

    plt.close(fig)

    fig = plot_actual_vs_predicted_scatter(
        y_true=y,
        y_pred=oof_predictions,
        title="OOF Actual vs Predicted"
    )

    mlflow.log_figure(
        fig,
        "plots/oof_actual_vs_predicted.png"
    )

    plt.close(fig)

    print('Logging Done')

Training:   0%|          | 0/5 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001645 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10554
[LightGBM] [Info] Number of data points in the train set: 16504, number of used features: 49
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 84.506907
Fold 01 | RMSE: 15.8556 | MAE: 11.0947 | R2 score: 0.8379 | S-score: 37051.6417
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001898 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10556
[LightGBM] [Info] Number of data points in the train set: 16505, number of used features: 49
[LightGBM] [Info] Using GOSS
[LightGBM] [Info] Start training from score 84.976068
Fold 02 | RMSE: 15.4496 | MAE: 10.9078 | R2 score: 0.8501 | S-score: 31562.0053
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002211